# 1 · Applicant EDA

Explore the observed cohort, segment repayment-difficulty rates and missingness. TARGET is a repayment-difficulty proxy, not a measured loss. These tables describe all applicants; predictive selection uses training rows only in the next notebook.

In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import display, Image, Markdown

# The executor supplies these paths. Interactive runs default to local exports.
ROOT = Path(os.environ.get('HOME_CREDIT_PROJECT_ROOT', str(Path.cwd()))).resolve()
if not (ROOT / 'src' / 'analysis.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis import ensure_analysis, load_features, eda_tables

INPUT = Path(os.environ.get('HOME_CREDIT_FEATURES', str(ROOT / 'exports' / 'applicant_features.csv')))
OUTPUT = Path(os.environ.get('HOME_CREDIT_ANALYSIS', str(ROOT / 'outputs' / 'analysis')))
STATUS = os.environ.get('HOME_CREDIT_DATA_STATUS', 'real')
LGD = float(os.environ.get('HOME_CREDIT_LGD', '0.45'))
SEED = int(os.environ.get('HOME_CREDIT_SEED', '42'))
manifest = ensure_analysis(INPUT, OUTPUT, data_status=STATUS, lgd=LGD, seed=SEED)
get_ipython().run_line_magic('matplotlib', 'inline')
display(Markdown(f'**Data status: {manifest["data_status"].upper()}**. '
    + ('Synthetic pipeline validation; no Home Credit conclusions.' if STATUS != 'real'
       else 'Computed from the supplied accepted-cohort export.')))


In [ ]:
frame = load_features(INPUT)
eda = eda_tables(frame, OUTPUT)
display(eda['overall'])
display(eda['nulls'].sort_values('null_pct', ascending=False).head(20))

## Segment rates

Wilson 95% intervals show rate uncertainty. Small segments can have noisy rates; compare sample sizes before interpreting differences. This is descriptive exploration, not a decision rule fitted to the test outcomes.

In [ ]:
segments = eda['segments']
display(segments.sort_values(['feature', 'bad_rate'], ascending=[True, False]).head(40))

In [ ]:
import matplotlib.pyplot as plt
age = segments[segments.feature == 'age_band']
if not age.empty:
    ax = age.plot.bar(x='level', y='bad_rate', legend=False, figsize=(7, 4), title='Observed TARGET rate by age band')
    ax.set_ylabel('Repayment-difficulty rate')
    ax.set_xlabel('Age band')
    plt.tight_layout()
    plt.show()

## Scope

Only application_train outcomes are analyzed. Raw data retain test-population historical records; SQL filters intermediate features to the observed training applicants. Missing numeric history measurements remain missing, while genuinely absent history counts are zero. Full raw null/sentinel/orphan checks are in the separate DQ exports.